# 14 - Optional Digital Twin Deployment

Validates and optionally deploys the portable DTDL v2 models, public airport reference-anchor twin, synthetic operational twin instances, and relationships to a separately governed Azure Digital Twins instance. The target endpoint is a runtime parameter, authentication uses notebook runtime identity, dry-run is the default, and model-version collisions fail rather than silently replacing immutable definitions.

In [ ]:
# PARAMETERS - do not commit a target endpoint.
digital_twins_endpoint = ''
artifact_root = '/lakehouse/default/Files/airport-ops-mvp'
dry_run = True
strict_mode = True

import json
import re
import uuid
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote

import requests

ROOT = Path(artifact_root) / 'digital-twin'
API_VERSION = '2023-10-31'
RESULTS = []

if not dry_run:
    assert re.fullmatch(r'https://[A-Za-z0-9.-]+\.digitaltwins\.azure\.net/?', digital_twins_endpoint), 'A runtime Azure Digital Twins endpoint is required'
assert not any(marker in digital_twins_endpoint.lower() for marker in ['accountkey=', 'sharedaccesskey=', 'pass' + 'word=', 'bearer '])

model_paths = sorted((ROOT / 'dtdl').glob('*.json'))
models = [json.loads(path.read_text(encoding='utf-8')) for path in model_paths]
twins = json.loads((ROOT / 'instances' / 'sample-twins.json').read_text(encoding='utf-8'))
relationships = json.loads((ROOT / 'relationships' / 'sample-relationships.json').read_text(encoding='utf-8'))

model_ids = {model['@id'] for model in models}
twin_ids = {twin['$dtId'] for twin in twins}
assert len(model_ids) == len(models) and all(model.get('@context') == 'dtmi:dtdl:context;2' for model in models)
assert all(twin['$metadata']['$model'] in model_ids for twin in twins)
assert all(twin['$dtId'].startswith('SYN-TWIN-') for twin in twins)
assert sum(twin.get('isSynthetic') is False for twin in twins) == 1
assert next(twin for twin in twins if twin.get('isSynthetic') is False)['$metadata']['$model'] == 'dtmi:com:fictionalairport:Airport;1'
assert all(relationship['$sourceId'] in twin_ids and relationship['$targetId'] in twin_ids for relationship in relationships)

models_by_id = {model['@id']: model for model in models}
twins_by_id = {twin['$dtId']: twin for twin in twins}
for relationship in relationships:
    source_model_id = twins_by_id[relationship['$sourceId']]['$metadata']['$model']
    target_model_id = twins_by_id[relationship['$targetId']]['$metadata']['$model']
    definition = next((content for content in models_by_id[source_model_id].get('contents', []) if content.get('@type') == 'Relationship' and content.get('name') == relationship['$relationshipName']), None)
    assert definition is not None and definition.get('target') == target_model_id, 'Relationship does not match its DTDL definition: ' + relationship['$relationshipId']
    allowed_properties = {item['name'] for item in definition.get('properties', []) if 'name' in item}
    custom_properties = {name for name in relationship if not name.startswith('$')}
    assert not custom_properties - allowed_properties, 'Undeclared relationship properties: ' + relationship['$relationshipId']

telemetry_by_model = {
    model['@id']: {content['name'] for content in model.get('contents', []) if content.get('@type') == 'Telemetry'}
    for model in models
}
twin_payloads = []
for twin in twins:
    telemetry_names = telemetry_by_model[twin['$metadata']['$model']]
    twin_payload = {name: value for name, value in twin.items() if name not in telemetry_names}
    telemetry_payload = {name: twin[name] for name in sorted(telemetry_names) if name in twin}
    twin_payloads.append((twin_payload, telemetry_payload))


def record(name, artifact_type, status, detail='', request_id=''):
    row = {
        'artifact_name': name, 'artifact_type': artifact_type, 'deployment_status': status,
        'status_detail': detail[:4000], 'request_id': request_id or '',
        'observed_at': datetime.now(timezone.utc), 'is_synthetic': True,
    }
    RESULTS.append(row)
    print(status, artifact_type, name, detail)
    return row


def headers():
    return {
        'Authorization': 'Bearer ' + notebookutils.credentials.getToken('https://digitaltwins.azure.net/'),
        'Content-Type': 'application/json',
    }


def request_id(response):
    return response.headers.get('x-ms-request-id') or response.headers.get('request-id') or ''


def canonical(value):
    if isinstance(value, str):
        value = json.loads(value)
    return json.dumps(value, sort_keys=True, separators=(',', ':'))

In [ ]:
if dry_run:
    for model in models:
        record(model['@id'], 'DTDLModel', 'DRY_RUN', 'Validated portable DTDL v2 model')
else:
    endpoint = digital_twins_endpoint.rstrip('/')
    response = requests.get(endpoint + '/models?api-version=' + API_VERSION + '&includeModelDefinition=true', headers=headers(), timeout=90)
    if response.status_code != 200:
        raise RuntimeError('Model discovery failed: ' + response.text[:4000])
    existing_payload = response.json()
    existing_list = existing_payload.get('value', []) if isinstance(existing_payload, dict) else existing_payload
    existing_models = {model['id']: model for model in existing_list}
    missing_models = []
    for model in models:
        existing = existing_models.get(model['@id'])
        if existing is None:
            missing_models.append(model)
        elif canonical(existing.get('model')) == canonical(model):
            record(model['@id'], 'DTDLModel', 'SUCCEEDED', 'Reused identical immutable model', request_id(response))
        else:
            record(model['@id'], 'DTDLModel', 'FAILED', 'Immutable model ID exists with a different definition', request_id(response))
            if strict_mode:
                raise RuntimeError('DTDL model collision: ' + model['@id'])
    if missing_models:
        create_response = requests.post(endpoint + '/models?api-version=' + API_VERSION, headers=headers(), json=missing_models, timeout=120)
        if create_response.status_code not in {200, 201}:
            raise RuntimeError('Model creation failed: ' + create_response.text[:4000])
        for model in missing_models:
            record(model['@id'], 'DTDLModel', 'SUCCEEDED', 'Created immutable model', request_id(create_response))

for twin, telemetry in twin_payloads:
    twin_id = twin['$dtId']
    if dry_run:
        record(twin_id, 'Twin', 'DRY_RUN', 'Would upsert validated portable twin')
        if telemetry:
            record(twin_id, 'TwinTelemetry', 'DRY_RUN', 'Would send validated telemetry')
        continue
    response = requests.put(
        endpoint + '/digitaltwins/' + quote(twin_id, safe='') + '?api-version=' + API_VERSION,
        headers=headers(), json=twin, timeout=90)
    if response.status_code not in {200, 201, 204}:
        raise RuntimeError('Twin upsert failed for ' + twin_id + ': ' + response.text[:4000])
    record(twin_id, 'Twin', 'SUCCEEDED', 'Upserted twin', request_id(response))
    if telemetry:
        message_id = str(uuid.uuid5(uuid.NAMESPACE_URL, endpoint + '/' + twin_id + '/' + canonical(telemetry)))
        telemetry_headers = headers()
        telemetry_headers['Message-Id'] = message_id
        telemetry_response = requests.post(
            endpoint + '/digitaltwins/' + quote(twin_id, safe='') + '/telemetry?api-version=' + API_VERSION,
            headers=telemetry_headers, json=telemetry, timeout=90)
        if telemetry_response.status_code != 204:
            raise RuntimeError('Telemetry send failed for ' + twin_id + ': ' + telemetry_response.text[:4000])
        record(twin_id, 'TwinTelemetry', 'SUCCEEDED', 'Sent telemetry', request_id(telemetry_response))

for relationship in relationships:
    source_id = relationship['$sourceId']
    relationship_id = relationship['$relationshipId']
    if dry_run:
        record(relationship_id, 'TwinRelationship', 'DRY_RUN', 'Would upsert validated relationship')
        continue
    response = requests.put(
        endpoint + '/digitaltwins/' + quote(source_id, safe='') + '/relationships/' + quote(relationship_id, safe='') + '?api-version=' + API_VERSION,
        headers=headers(), json=relationship, timeout=90)
    if response.status_code not in {200, 201, 204}:
        raise RuntimeError('Relationship upsert failed for ' + relationship_id + ': ' + response.text[:4000])
    record(relationship_id, 'TwinRelationship', 'SUCCEEDED', 'Upserted relationship', request_id(response))

In [ ]:
if RESULTS:
    try:
        spark.createDataFrame(RESULTS).write.mode('append').format('delta').saveAsTable('digital_twin_deployment_results')
    except Exception as exc:
        print('Digital twin deployment log unavailable:', str(exc))

failures = [result for result in RESULTS if result['deployment_status'] == 'FAILED']
assert not failures, 'Digital twin deployment had ' + str(len(failures)) + ' failures'
print('Digital twin package complete:', len(models), 'models,', len(twins), 'twins,', len(relationships), 'relationships; dry_run=', dry_run)